In [ ]:
from fp_qsim import hello

output = hello()
print(output)

Hello from fp-qsim!


In [39]:
# mocked simulator for testing purposes
# from IPython.terminal.pt_inputhooks.osx import n
# from PIL.Image import new
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from dataclasses import dataclass
import numpy as np
from qiskit.quantum_info import Operator


@dataclass
class MockSimulator(AerSimulator):
	def __init__(self) -> None:
		super().__init__()
		self._statevector_simulator = AerSimulator(method='statevector')

	def run(self, circuits: QuantumCircuit, shots: int) -> object:  # ty:ignore[invalid-method-override]
		result = self._statevector_simulator.run(circuits, shots=shots).result()
		return result


@dataclass
class CustomSimulator:
	def run(self, circuit: QuantumCircuit, shots: int = 1024) -> np.ndarray:
		"""Apply each gate tensor to the state tensor using numpy.einsum."""
		n_qubits = circuit.num_qubits

		# Tensor axes are ordered as [q_{n-1}, ..., q_0] so flattening matches Qiskit's basis order.
		state = np.zeros([2] * n_qubits, dtype=complex)
		state[(0,) * n_qubits] = 1.0

		for instruction in circuit.data:
			gate = instruction.operation
			qubits = [circuit.find_bit(q).index for q in instruction.qubits][::-1]
			n_gate_qubits = len(qubits)

			# Gate tensor has indices: [out_0, ..., out_k-1, in_0, ..., in_k-1].
			gate_tensor = Operator(gate).data.reshape([2] * (2 * n_gate_qubits))
			# print(Operator(gate).data)

			# Map logical qubits to tensor axes in state.
			state_axes = list(range(n_qubits))[::-1]
			qubit_axes = list(qubits)  # [n_qubits - 1 - q for q in qubits]

			out_axes = state_axes.copy()
			new_gate_out_axes = list(range(n_qubits, n_qubits + n_gate_qubits))
			for i, axis in enumerate(qubit_axes):
				out_axes[n_qubits - 1 - axis] = new_gate_out_axes[i]
			# print(new_gate_out_axes, qubit_axes, state_axes, out_axes)

			# Contract gate inputs with target state axes and keep updated state axes.
			state = np.einsum(
				gate_tensor,
				new_gate_out_axes + qubit_axes,
				state,
				state_axes,
				out_axes,
				optimize=False,
			)

		return state.reshape(-1)


ghz_circ = QuantumCircuit(2)
ghz_circ.x(0)
ghz_circ.cx(0, 1)

custom_state = CustomSimulator().run(ghz_circ)

ref_sim = AerSimulator(method='statevector')
ghz_circ.save_statevector()
ref_state = ref_sim.run(ghz_circ).result().get_statevector()

print('Custom statevector:', custom_state)
print('Reference statevector:', ref_state)
print('Match:', np.allclose(custom_state, ref_state))

"""old_indices= [i for i in range(n_qubits)]
new_indices = old_indices.copy()
phi = np.einsum('ij,j->i', gate_tensor, state.flatten())"""

Custom statevector: [0.+0.j 0.+0.j 0.+0.j 1.+0.j]
Reference statevector: Statevector([0.+0.j, 0.+0.j, 0.+0.j, 1.+0.j],
            dims=(2, 2))
Match: True


"old_indices= [i for i in range(n_qubits)]\nnew_indices = old_indices.copy()\nphi = np.einsum('ij,j->i', gate_tensor, state.flatten())"

In [9]:
arr = np.array([1, 2, 3, 4])
print(arr.reshape(2, 2)[0, 1])

2
